In [203]:
from pathlib import Path
from dataclasses import dataclass

csv_dir = Path(r'C:\Users\user\Desktop\div\notebooks\20261123\bank')
ACCOUNTS_FILE = csv_dir / "accounts.csv"
TRANSACTIONS_FILE = csv_dir / "transactions.csv"

@dataclass
class Transaction:
	number : str
	account : str
	date : str
	amount : float
	category : str
	note : str

	def __str__(self):
		return (
			f"{self.number},"
			f"{self.account},"
			f"{self.date},"
			f"{self.amount:.2f},"
			f"{self.category},"
			f"{self.note}"
			)

class Account:
	def __init__(self, number, name, owner, currency, state):
		self.name = name
		self.number = number
		self.owner = owner
		self.currency = currency
		self.state = state
		self.transactions = []

	@property
	def balance(self):
		return sum([t.amount for t in self.transactions])


	def can_pay(self):
		return (self.state == 'active') and (self.balance > 0)

	def __repr__(self):
		return (
			f"<Account: {self.number}, "
			f"transactions: {len(self.transactions)}, "
			f"balance: {self.balance} AZN"
			">")

def load():
	
	with ACCOUNTS_FILE.open('r', encoding='UTF-8') as file:
		lines = file.read().splitlines()[1:]
	accounts = {}
	for line in lines :
		number, name, owner, currency, state = line.split(',')
		a = Account(number, name, owner, currency, state)
		accounts[number] = a

	transactions = {}
	with TRANSACTIONS_FILE.open('r', encoding='UTF-8') as file:
		lines = file.read().splitlines()[1:]
	for line in lines:
		t_number,a_number,date,amount,category,note = line.split(',')
		t = Transaction(t_number,a_number,date,float(amount),category,note)
		transactions[t_number] = t
		accounts[a_number].transactions.append(t)

	return accounts, transactions

accounts, transactions, = load()
a1 = accounts['A200']
a1.can_pay()


True

In [204]:
import datetime

def find_max_transaction_number() -> int:
	max_t_number = max([int(k[1:]) for k in transactions.keys()])
	return max_t_number

def add_transaction(t):
	with TRANSACTIONS_FILE.open('a', encoding='UTF-8') as file:
		file.write('\n' + str(t))

def transfer(accounts, source, target, amount: float) -> bool:
	if source not in accounts or target not in accounts:
		return False
	elif not accounts[source].can_pay():
		return False
	elif accounts[target].state != 'active':
		return False
	elif accounts[source].balance < amount:
		return False
	elif accounts[source].currency != accounts[target].currency:
		return False

	max_t_number = find_max_transaction_number()
	source_account = accounts[source]
	T_source = Transaction(
		number = f"T{max_t_number+1:0>3}", 
		account = source,
		date = datetime.datetime.now().strftime("%Y-%m-%d"),
		amount = -amount,
		category= 'transfer',
		note='X'
		)
	source_account.transactions.append(T_source)
	add_transaction(T_source)

	target_account = accounts[target]
	T_target = Transaction(
		number = f"T{max_t_number+2:0>3}",
		account = target,
		date = datetime.datetime.now().strftime("%Y-%m-%d"),
		amount = amount,
		category= 'transfer',
		note='X'
		)
	target_account.transactions.append(T_target)
	add_transaction(T_target)
	
	return True

transfer(accounts, 'A100', 'A200', 10)

True

In [206]:
accounts['A100'].balance

20.0

In [ ]:
# class Account:
# 	def __init__(self, number, name, owner, currency, state):
# 		self.name = name
# 		self.number = number
# 		self.owner = owner
# 		self.currency = currency
# 		self.state = state

from dataclasses import dataclass

@dataclass
class Account:
	number: str
	name: str
	owner: str
	currency: str
	state: str

a1 = Account('A100','salary','Əli','AZN','active')
a1.name

'salary'